## finding errors in JN .ipynb files -berkeley_pipeline

In [4]:
#cell 1
# CELL: Compare Two Versions of Pipeline Notebook

import json
from pathlib import Path
from datetime import datetime

file1 = Path('/Users/johngage/berkeley-data/berkeley_open_data_pipeline.ipynb')
file2 = Path('/Users/johngage/Downloads/berkeley_open_data_pipeline.ipynb')

print("📊 COMPARING TWO VERSIONS OF PIPELINE NOTEBOOK")
print("="*70)

for filepath in [file1, file2]:
    print(f"\n📁 {filepath.parent.name}/{filepath.name}")
    print("-"*70)
    
    if filepath.exists():
        # File metadata
        stat = filepath.stat()
        print(f"   Size: {stat.st_size:,} bytes ({stat.st_size / 1024:.1f} KB)")
        print(f"   Modified: {datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Created: {datetime.fromtimestamp(stat.st_ctime).strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Parse notebook
        with open(filepath, 'r') as f:
            nb = json.load(f)
        
        cells = nb.get('cells', [])
        code_cells = [c for c in cells if c.get('cell_type') == 'code']
        markdown_cells = [c for c in cells if c.get('cell_type') == 'markdown']
        
        print(f"\n   📝 Total cells: {len(cells)}")
        print(f"      Code: {len(code_cells)}")
        print(f"      Markdown: {len(markdown_cells)}")
        
        # Check execution counts
        executed = [c for c in code_cells if c.get('execution_count') is not None]
        if executed:
            exec_counts = [c['execution_count'] for c in executed]
            print(f"   🔢 Executed cells: {len(executed)}")
            print(f"      Execution range: {min(exec_counts)} to {max(exec_counts)}")
        else:
            print(f"   🔢 No cells executed (clean notebook)")
        
        # Check for CSV output
        writes_final_csv = False
        writes_any_csv = []
        
        for i, cell in enumerate(code_cells):
            source = ''.join(cell.get('source', []))
            
            if 'housing_projects_final_complete.csv' in source and 'to_csv' in source:
                writes_final_csv = True
                print(f"\n   ✅ Cell #{i+1} WRITES housing_projects_final_complete.csv")
            
            if '.to_csv' in source and 'housing' in source.lower():
                for line in source.split('\n'):
                    if '.to_csv' in line:
                        writes_any_csv.append((i+1, line.strip()[:60]))
        
        if writes_final_csv:
            print(f"   ✅ Creates final CSV!")
        elif writes_any_csv:
            print(f"\n   📝 Writes other housing CSVs ({len(writes_any_csv)} cells):")
            for cell_num, line in writes_any_csv[:3]:
                print(f"      Cell #{cell_num}: {line}...")
        else:
            print(f"   ❌ Doesn't write housing CSV")
        
        # Check for error patterns
        error_patterns = ['Error', 'Exception', 'failed', 'fix', 'correction', 'oops', 'wrong']
        cells_with_errors = 0
        
        for cell in code_cells:
            source = ''.join(cell.get('source', [])).lower()
            if any(pattern.lower() in source for pattern in error_patterns):
                cells_with_errors += 1
        
        if cells_with_errors > 0:
            print(f"\n   ⚠️  Cells mentioning errors/fixes: {cells_with_errors}")
        else:
            print(f"\n   ✨ Clean notebook (no error mentions)")
    
    else:
        print(f"   ❌ File not found")

print("\n" + "="*70)
print("\n💡 RECOMMENDATION:")
print("   Compare the details above to decide which version to use")

📊 COMPARING TWO VERSIONS OF PIPELINE NOTEBOOK

📁 berkeley-data/berkeley_open_data_pipeline.ipynb
----------------------------------------------------------------------
   Size: 265,276 bytes (259.1 KB)
   Modified: 2026-01-07 17:28:17
   Created: 2026-01-07 17:28:17

   📝 Total cells: 22
      Code: 16
      Markdown: 6
   🔢 Executed cells: 14
      Execution range: 2 to 43

   📝 Writes other housing CSVs (6 cells):
      Cell #4: df.to_csv('/Users/johngage/berkeley-data/berkeley_datasets.c...
      Cell #6: df.to_csv('/Users/johngage/berkeley-data/all_berkeley_datase...
      Cell #6: relevant.to_csv('/Users/johngage/berkeley-data/relevant_data...

   ⚠️  Cells mentioning errors/fixes: 7

📁 Downloads/berkeley_open_data_pipeline.ipynb
----------------------------------------------------------------------
   Size: 146,863 bytes (143.4 KB)
   Modified: 2026-01-08 18:16:44
   Created: 2026-01-08 18:16:44

   📝 Total cells: 16
      Code: 14
      Markdown: 2
   🔢 Executed cells: 13
      

In [5]:
# cell 2
# CELL: Compare Data Processing Steps

import json
from pathlib import Path

file1 = Path('/Users/johngage/berkeley-data/berkeley_open_data_pipeline.ipynb')
file2 = Path('/Users/johngage/Downloads/berkeley_open_data_pipeline.ipynb')

print("🔍 COMPARING DATA PROCESSING COMPLETENESS")
print("="*70)

key_steps = [
    'download',
    'geocod',
    'clean',
    'merge',
    'filter',
    'validate',
    'export',
    'to_csv'
]

for filepath in [file1, file2]:
    print(f"\n📁 {filepath.parent.name}/{filepath.name}")
    print("-"*70)
    
    with open(filepath, 'r') as f:
        nb = json.load(f)
    
    # Check which processing steps are present
    all_code = []
    for cell in nb.get('cells', []):
        if cell.get('cell_type') == 'code':
            all_code.append(''.join(cell.get('source', [])).lower())
    
    full_text = '\n'.join(all_code)
    
    print("   Processing steps found:")
    for step in key_steps:
        count = full_text.count(step)
        if count > 0:
            print(f"      ✅ {step}: {count} occurrences")
        else:
            print(f"      ❌ {step}: not found")

print("\n" + "="*70)

🔍 COMPARING DATA PROCESSING COMPLETENESS

📁 berkeley-data/berkeley_open_data_pipeline.ipynb
----------------------------------------------------------------------
   Processing steps found:
      ❌ download: not found
      ❌ geocod: not found
      ❌ clean: not found
      ❌ merge: not found
      ✅ filter: 11 occurrences
      ❌ validate: not found
      ❌ export: not found
      ✅ to_csv: 6 occurrences

📁 Downloads/berkeley_open_data_pipeline.ipynb
----------------------------------------------------------------------
   Processing steps found:
      ❌ download: not found
      ✅ geocod: 1 occurrences
      ✅ clean: 47 occurrences
      ❌ merge: not found
      ✅ filter: 7 occurrences
      ❌ validate: not found
      ✅ export: 8 occurrences
      ✅ to_csv: 1 occurrences



In [6]:
# Cell 3
# CELL: Check where files will be saved

import os
from pathlib import Path

print("📂 WORKING DIRECTORY INFO")
print("="*70)

# Current working directory
cwd = os.getcwd()
print(f"\nCurrent working directory: {cwd}")

# Where THIS notebook is located
notebook_dir = Path.cwd()
print(f"Notebook running from: {notebook_dir}")

# Check if they're the same
if '/Downloads' in str(cwd):
    print("\n⚠️  WARNING: Running from Downloads!")
    print("   Files will be saved to Downloads folder")
elif '/berkeley-data' in str(cwd):
    print("\n✅ Running from berkeley-data")
    print("   Files will be saved here")

# Where would a CSV be saved?
test_file = Path('housing_projects_final_complete.csv')
print(f"\nIf you save a CSV, it goes to:")
print(f"   {test_file.absolute()}")

print("\n" + "="*70)

📂 WORKING DIRECTORY INFO

Current working directory: /Users/johngage/berkeley-data
Notebook running from: /Users/johngage/berkeley-data

✅ Running from berkeley-data
   Files will be saved here

If you save a CSV, it goes to:
   /Users/johngage/berkeley-data/housing_projects_final_complete.csv

